In [1]:
import os
import re
import certifi
import urllib3
import requests
import pandas as pd

# Setup SSL certificates for secure requests
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['SSL_CERT_FILE'] = certifi.where()
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Common helper functions
def clean_html(raw):
    if not raw:
        return ""
    text = re.sub(r"<[^>]+>", " ", str(raw))
    text = text.replace("&nbsp;", " ").replace("\u2019", "'")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def to_date(published_at):
    return (published_at or "").split("T")[0]

def get_field(item, field):
    if "attributes" in item:
        return item["attributes"].get(field, "")
    return item.get(field, "")

# Initialize global tracking list
all_rows = []
DOMAIN_CODE = "SEC"

In [2]:
# ==========================================
# 1. FETCH FROM NTSA (Road Safety)
# ==========================================
DOMAIN_NTSA = "Security and Safety"
SUB_CATEGORY_NTSA = "Road Safety"
SOURCE_NTSA = "NTSA (National Transport and Safety Authority)"

try:
    resp_ntsa = requests.get(
        "https://ntsa.go.ke/api/news",
        headers={"User-Agent": "Mozilla/5.0"},
        verify=False
    )
    items_ntsa = resp_ntsa.json().get("data", [])
    print(f"Got {len(items_ntsa)} items from NTSA")
    
    for item in items_ntsa:
        all_rows.append({
            "Domain": DOMAIN_NTSA,
            "Sub_Category": SUB_CATEGORY_NTSA,
            "English": clean_html(item.get("excerpt", "")),
            "Kiswahili": "",
            "Target_Language": "",
            "Source": SOURCE_NTSA,
            "Date": to_date(item.get("published_at", "")),
            "Metadata": f"tone=advisory; platform=website; category={item.get('category', '')}",
        })
except Exception as e:
    print(f"Error fetching NTSA data: {e}")

# ==========================================
# 2. FETCH FROM NC4 (Cybersecurity)
# ==========================================
DOMAIN_NC4 = "Security and Safety"
SUB_CATEGORY_NC4 = "Cybersecurity"
SOURCE_NC4 = "NC4 (National Computer and Cybercrime Coordination Committee)"

url_nc4 = "https://studio.nc4.go.ke/api/news-items"
params_nc4 = {
    "filters[type][$eq]": "alert",
    "sort": "publishedAt:desc",
    "pagination[pageSize]": 100,
}

try:
    resp_nc4 = requests.get(url_nc4, params=params_nc4, headers={"User-Agent": "Mozilla/5.0"})
    resp_nc4.raise_for_status()
    items_nc4 = resp_nc4.json().get("data", [])
    print(f"Got {len(items_nc4)} alert items from NC4")
    
    for item in items_nc4:
        excerpt = (
            get_field(item, "excerpt")
            or get_field(item, "description")
            or get_field(item, "summary")
        )
        all_rows.append({
            "Domain": DOMAIN_NC4,
            "Sub_Category": SUB_CATEGORY_NC4,
            "English": clean_html(excerpt),
            "Kiswahili": "",
            "Target_Language": "",
            "Source": SOURCE_NC4,
            "Date": to_date(get_field(item, "publishedAt")),
            "Metadata": "tone=advisory; platform=website; alert_type=cybersecurity",
        })
except Exception as e:
    print(f"Error fetching NC4 data: {e}")

# ==========================================
# 3. BUILD DATAFRAME & EXPORT TO EXCEL
# ==========================================
if all_rows:
    df = pd.DataFrame(all_rows)
    
    # Inject contiguous sequence numbers (SEC_001, SEC_002, etc.) across all combined entries
    df.insert(0, "PSA_ID", [f"{DOMAIN_CODE}_{i:03d}" for i in range(1, len(df) + 1)])
    
    # Enforce precise column ordering
    fields = ["PSA_ID", "Domain", "Sub_Category", "English", "Kiswahili",
              "Target_Language", "Source", "Date", "Metadata"]
    df = df[fields]
    
    # Save output spreadsheet
    output_file = "combined_psa_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\nSuccess! Saved {len(df)} total aggregated records to {output_file}")
else:
    print("\nNo rows gathered from either source. Excel file not created.")

Got 14 items from NTSA
Got 8 alert items from NC4

Success! Saved 22 total aggregated records to combined_psa_data.xlsx
